In [18]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict , Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [19]:
load_dotenv()
model = ChatGroq(model="llama-3.1-8b-instant",temperature=1.5)


In [20]:
class SentimentSchema(BaseModel):
    sentiment: Literal['positive','negative'] = Field(description='Sentiment of the review')

In [21]:
class DiagonsisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug","support", "other"] = Field(description="The category of the issue mentioned in the review")
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description="The emotional tone expressed by the user")
    urgency: Literal["low","medium", "high"] = Field(description="how urgent or critical the issue appears to be")
    

In [22]:
sentiment_model = model.with_structured_output(SentimentSchema)
diagnosis_model = model.with_structured_output(DiagonsisSchema)

In [23]:
class reviewState(TypedDict):
    review:str

    sentiment: Literal['positive','negative']
    diagnosis: dict
    response : str 

In [24]:
def find_sentiment(state: reviewState):
    prompt = f"""Analyze the sentiment of this review and respond with ONLY a function call:
    
    Review: {state["review"]}
    
    Call the function with either "positive" or "negative"."""
    sentiment = sentiment_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

def check_sentiment(state: reviewState)-> Literal['positive_response', 'run_diagnosis']:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

def run_diagnosis(state: reviewState):
    prompt = f"""Diagnose this negative review: {state['review']}\nReturn issue type, tone, and urgency."""
    response = diagnosis_model.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def positive_response(state: reviewState):
    prompt = f"""write a warm thank you message in response to this review: \n\n {state['review']} \n
    also, kindle ask the user to leave feedback on our website"""
    response = model.invoke(prompt).content
    return {"response": response}

def negative_response(state: reviewState):
    diagnosis = state["diagnosis"] # diagnosing the issue
    prompt = f"""you are a support assistant, the user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    write an empathetic, helpful resolution message"""
    response = model.invoke(prompt).content
    return {"response": response}

In [25]:

graph = StateGraph(reviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)
graph.add_node('positive_response',positive_response)


graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment )
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [32]:
initial_state = {
    "review": "I 've been using this app for about a rnonth now, and I must say, the user interface is incredibly clean and intuitive. Everything is exactly where you'd expect it to be. It's rare to find something that just works without needing a tutorial. Great job to the designi team!"
}

ans = workflow.invoke(initial_state)
ans['sentiment']

'positive'